# TAMUG HDC models on the OnlineHD and QuantHD paper datasets

This notebook is intentionally separate from the ImageNetV2 notebooks. It runs the HDC heads used in this repository on the tabular/image-vector benchmarks and experimental scale reported by:

- *OnlineHD: Robust, Efficient, and Single-Pass Online Learning Using Hyperdimensional System* (DATE 2021)
- *QuantHD: A Quantization Framework for Hyperdimensional Computing* (IEEE TCAD 2020)

The default `paper` mode uses **10,000 hypervector dimensions**, the papers' train/test sizes where the public data make them reproducible, and the project implementations of OnlineHD and QuantHD. The models are ours; this is therefore a controlled same-data/same-scale comparison, not a claim that every number must equal the papers' custom C++/FPGA implementations.

Exact, automatically downloaded public inputs:

| Notebook name | Paper name | Features | Classes | Train | Test | Split fidelity |
|---|---|---:|---:|---:|---:|---|
| `mnist` | MNIST | 784 | 10 | 60,000 | 10,000 | Official split |
| `isolet` | ISOLET | 617 | 26 | 6,238 | 1,559 | Official ISOLET 1-4 / ISOLET 5 split |
| `sa12` | UCIHAR in the papers | 561 | 12 | 6,213 | 1,554 | Deterministic stratified 80/20 split of the 7,767-sample HAPT training partition; the papers do not publish their random indices |

The papers' “UCIHAR” is the 12-class HAPT/SA12 variant, not the common 6-class UCI-HAR dataset (7,352/2,947). This notebook checks the 12-class counts so the two cannot be confused.

PAMAP2, FACE, PECAN, EXTRA, and CARDIO are supported through validated `.npz` adapters. Their exact processed matrices/splits are not supplied by the papers, and substituting a modern preprocessing recipe would no longer be the same setting. Put any author-compatible arrays in `data/paper_benchmarks/custom/<name>.npz`; the file must contain `x_train`, `y_train`, `x_test`, and `y_test`.

Data sources: [UCI ISOLET](https://archive.ics.uci.edu/dataset/54/isolet), [UCI HAPT](https://archive.ics.uci.edu/dataset/341/smartphone+based+recognition+of+human+activities+and+postural+transitions), [TorchVision MNIST](https://pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html), and the authors' [OnlineHD implementation](https://github.com/BIASLab-UCI/onlinehd).


## 1. Configuration

Run this cell first. `paper` is the default. Set the environment variable `PAPER_BENCHMARK_MODE=smoke` before starting Jupyter for a fast pipeline check with reduced data and dimensionality.

Paper-setting notes:

- OnlineHD accuracy experiments use `D = 10,000` and report both single-pass and iterative training. This notebook defaults to one adaptive pass and exposes `ONLINEHD_PASSES` for iterative runs. The learning rate `0.035` follows the authors' public implementation because the paper does not state a numeric value.
- QuantHD Section V-B reports 40 retraining iterations with `alpha = 0.05`; those are the defaults here. Section III-B separately describes a 5% validation stopping rule (`epsilon = 0.01`, maximum 30 iterations), so an opt-in toggle is provided for that protocol.
- The project QuantHD head is binary. The paper's ternary boundary experiment is not silently substituted for it.


In [ ]:
import os
from pathlib import Path

RUN_MODE = os.environ.get("PAPER_BENCHMARK_MODE", "paper").strip().lower()
if RUN_MODE not in {"paper", "smoke"}:
    raise ValueError("PAPER_BENCHMARK_MODE must be 'paper' or 'smoke'")

PROJECT_DIR = Path.cwd().resolve()
DATA_DIR = PROJECT_DIR / "data" / "paper_benchmarks"
ARTIFACT_DIR = PROJECT_DIR / "artifacts" / "paper_benchmarks"

# Public paper datasets that can be reconstructed without unpublished feature files.
DATASET_NAMES = ("mnist", "isolet", "sa12")
# Project classifiers. Add "baseline_hdc" for the one-pass prototype baseline.
MODEL_NAMES = ("onlinehd", "quanthd")

HYPERVECTOR_DIMENSIONS = 10_000
BATCH_SIZE = 256
RANDOM_SEED = 0
DEVICE_PREFERENCE = "auto"  # auto, cuda, mps, or cpu

ONLINEHD_PASSES = 1
ONLINEHD_LEARNING_RATE = 0.035

QUANTHD_RETRAIN_PASSES = 40
QUANTHD_LEARNING_RATE = 0.05
QUANTHD_USE_METHOD_STOPPING = False
QUANTHD_VALIDATION_FRACTION = 0.05
QUANTHD_CONVERGENCE_EPSILON = 0.01
QUANTHD_MAX_METHOD_PASSES = 30

REUSE_CHECKPOINTS = True
SAVE_ARTIFACTS = True
FORCE_REDOWNLOAD = False

# Smoke mode is for validation only; it is deliberately not a paper result.
MAX_TRAIN_SAMPLES = None
MAX_TEST_SAMPLES = None
if RUN_MODE == "smoke":
    DATASET_NAMES = ("isolet",)
    HYPERVECTOR_DIMENSIONS = 256
    BATCH_SIZE = 64
    MAX_TRAIN_SAMPLES = 256
    MAX_TEST_SAMPLES = 128
    QUANTHD_RETRAIN_PASSES = 1
    QUANTHD_USE_METHOD_STOPPING = False
    REUSE_CHECKPOINTS = False
    SAVE_ARTIFACTS = False

print(f"Mode: {RUN_MODE}")
print(f"Datasets: {DATASET_NAMES}")
print(f"Models: {MODEL_NAMES}")
print(f"D={HYPERVECTOR_DIMENSIONS:,}, batch_size={BATCH_SIZE}, seed={RANDOM_SEED}")


## 2. Imports, device selection, and reproducibility

This notebook needs PyTorch, TorchVision, NumPy, pandas, matplotlib, and tqdm. They are already used by this repository; the cell reports a direct installation command if something is missing.


In [ ]:
import gc
import hashlib
import json
import math
import random
import shutil
import subprocess
import time
import urllib.request
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone

try:
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torchvision
    from IPython.display import display
    from torch.utils.data import DataLoader, TensorDataset
    from tqdm.auto import tqdm
except ImportError as exc:
    raise ImportError(
        "Missing dependency. Install with: pip install torch torchvision numpy pandas matplotlib tqdm"
    ) from exc

def select_device(preference):
    preference = preference.lower()
    if preference == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    if preference == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable")
    if preference == "mps" and not (
        hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    ):
        raise RuntimeError("MPS was requested but is unavailable")
    return torch.device(preference)

DEVICE = select_device(DEVICE_PREFERENCE)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"PyTorch {torch.__version__}; TorchVision {torchvision.__version__}; device={DEVICE}")


## 3. Paper dataset manifest

The validation rules below prevent a custom file with the wrong feature count, class count, or split size from being labeled as a paper-matched run. `cardio` is listed but has no accepted exact shape because the QuantHD paper reports its accuracy without documenting the corresponding dataset row in Table II.


In [ ]:
PAPER_DATASETS = {
    "mnist":  {"features": 784, "classes": 10, "train": 60_000,  "test": 10_000,  "papers": "OnlineHD"},
    "isolet": {"features": 617, "classes": 26, "train": 6_238,   "test": 1_559,   "papers": "OnlineHD + QuantHD"},
    "sa12":   {"features": 561, "classes": 12, "train": 6_213,   "test": 1_554,   "papers": "OnlineHD + QuantHD"},
    "pamap2": {"features": 75,  "classes": 5,  "train": 611_142, "test": 101_582, "papers": "OnlineHD + QuantHD"},
    "face":   {"features": 608, "classes": 2,  "train": 522_441, "test": 2_494,   "papers": "OnlineHD + QuantHD"},
    "pecan":  {"features": 312, "classes": 3,  "train": 22_290,  "test": 5_574,   "papers": "OnlineHD"},
    "extra":  {"features": 225, "classes": 4,  "train": 146_869, "test": 16_343,  "papers": "QuantHD"},
    "cardio": {"features": None,"classes": None,"train": None,    "test": None,    "papers": "QuantHD Table III; split unspecified"},
}

manifest = pd.DataFrame(PAPER_DATASETS).T.reset_index(names="dataset")
manifest["automatic"] = manifest["dataset"].isin({"mnist", "isolet", "sa12"})
manifest["custom_npz"] = manifest["dataset"].map(
    lambda name: str(DATA_DIR / "custom" / f"{name}.npz") if name not in {"mnist", "isolet", "sa12"} else ""
)
display(manifest)


## 4. Download and load the datasets

Downloads are cached under `data/paper_benchmarks/`. Public data are validated against the paper's shapes before any smoke-mode limits are applied.


In [ ]:
ISOLET_URL = "https://archive.ics.uci.edu/static/public/54/isolet.zip"
HAPT_URL = "https://archive.ics.uci.edu/static/public/341/smartphone+based+recognition+of+human+activities+and+postural+transitions.zip"

@dataclass
class BenchmarkData:
    name: str
    x_train: torch.Tensor
    y_train: torch.Tensor
    x_test: torch.Tensor
    y_test: torch.Tensor
    fidelity_note: str

    @property
    def n_features(self):
        return int(self.x_train.shape[1])

    @property
    def n_classes(self):
        return int(torch.unique(torch.cat([self.y_train, self.y_test])).numel())

def download_file(url, destination):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 0 and not FORCE_REDOWNLOAD:
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(destination.suffix + ".part")
    if partial.exists():
        partial.unlink()
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    print(f"Downloading {url}")
    with urllib.request.urlopen(request) as response, partial.open("wb") as stream:
        shutil.copyfileobj(response, stream)
    partial.replace(destination)
    return destination

def safe_extract_zip(archive, destination):
    archive, destination = Path(archive), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(archive) as zf:
        for member in zf.infolist():
            target = (destination / member.filename).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f"Unsafe path in archive: {member.filename}")
        zf.extractall(destination)

def atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)

def decompress_unix_compress(source):
    source = Path(source)
    destination = source.with_suffix("")
    if destination.exists():
        return destination
    gzip_binary = shutil.which("gzip")
    if gzip_binary is None:
        raise RuntimeError(
            f"{source.name} uses legacy .Z compression and 'gzip' is unavailable. "
            "Install gzip/uncompress, then rerun this cell."
        )
    subprocess.run([gzip_binary, "-dk", str(source)], check=True)
    return destination

def stratified_split_exact(labels, test_size, seed):
    labels = labels.cpu().numpy()
    rng = np.random.default_rng(seed)
    classes, counts = np.unique(labels, return_counts=True)
    raw = counts * (test_size / len(labels))
    allocations = np.floor(raw).astype(int)
    remainder = int(test_size - allocations.sum())
    order = np.argsort(-(raw - allocations))
    allocations[order[:remainder]] += 1
    test_parts = []
    for cls, count in zip(classes, allocations):
        indices = np.flatnonzero(labels == cls)
        rng.shuffle(indices)
        test_parts.append(indices[:count])
    test_indices = np.sort(np.concatenate(test_parts))
    train_mask = np.ones(len(labels), dtype=bool)
    train_mask[test_indices] = False
    return torch.from_numpy(np.flatnonzero(train_mask)), torch.from_numpy(test_indices)

def validate_full_dataset(data):
    spec = PAPER_DATASETS[data.name]
    if spec["features"] is not None and data.n_features != spec["features"]:
        raise ValueError(f"{data.name}: expected {spec['features']} features, got {data.n_features}")
    if spec["classes"] is not None and data.n_classes != spec["classes"]:
        raise ValueError(f"{data.name}: expected {spec['classes']} classes, got {data.n_classes}")
    if spec["train"] is not None and len(data.x_train) != spec["train"]:
        raise ValueError(f"{data.name}: expected {spec['train']} train rows, got {len(data.x_train)}")
    if spec["test"] is not None and len(data.x_test) != spec["test"]:
        raise ValueError(f"{data.name}: expected {spec['test']} test rows, got {len(data.x_test)}")
    for split, x, y in (("train", data.x_train, data.y_train), ("test", data.x_test, data.y_test)):
        if x.ndim != 2 or y.ndim != 1 or len(x) != len(y):
            raise ValueError(f"{data.name} {split}: invalid tensor shapes {tuple(x.shape)}, {tuple(y.shape)}")
        if not torch.isfinite(x).all():
            raise ValueError(f"{data.name} {split}: features contain NaN or infinity")
        if y.min().item() != 0 or y.max().item() != data.n_classes - 1:
            raise ValueError(f"{data.name} {split}: labels must be contiguous and zero-based")

def stratified_limit(x, y, maximum, seed):
    if maximum is None or maximum >= len(y):
        return x, y
    classes = torch.unique(y).tolist()
    if maximum < len(classes):
        raise ValueError(f"Limit {maximum} is smaller than the {len(classes)} classes")
    generator = torch.Generator().manual_seed(seed)
    chosen = []
    for cls in classes:
        class_indices = torch.nonzero(y == cls, as_tuple=False).flatten()
        chosen.append(class_indices[torch.randperm(len(class_indices), generator=generator)[0:1]])
    already = torch.cat(chosen)
    mask = torch.ones(len(y), dtype=torch.bool)
    mask[already] = False
    remaining = torch.nonzero(mask, as_tuple=False).flatten()
    remaining = remaining[torch.randperm(len(remaining), generator=generator)]
    selected = torch.cat([already, remaining[: maximum - len(already)]])
    selected = selected[torch.randperm(len(selected), generator=generator)]
    return x[selected], y[selected]

def apply_run_limits(data):
    x_train, y_train = stratified_limit(data.x_train, data.y_train, MAX_TRAIN_SAMPLES, RANDOM_SEED + 10)
    x_test, y_test = stratified_limit(data.x_test, data.y_test, MAX_TEST_SAMPLES, RANDOM_SEED + 20)
    return BenchmarkData(data.name, x_train, y_train, x_test, y_test, data.fidelity_note)


In [ ]:
def load_mnist():
    root = DATA_DIR / "mnist"
    train = torchvision.datasets.MNIST(root=root, train=True, download=True)
    test = torchvision.datasets.MNIST(root=root, train=False, download=True)
    return BenchmarkData(
        "mnist",
        train.data.reshape(len(train), -1).float().div_(255.0),
        train.targets.long(),
        test.data.reshape(len(test), -1).float().div_(255.0),
        test.targets.long(),
        "Exact official MNIST train/test split.",
    )

def load_isolet():
    root = DATA_DIR / "isolet"
    cache = root / "processed.pt"
    if cache.exists() and not FORCE_REDOWNLOAD:
        payload = torch.load(cache, map_location="cpu", weights_only=True)
        return BenchmarkData("isolet", **payload)
    archive = download_file(ISOLET_URL, root / "isolet.zip")
    safe_extract_zip(archive, root / "raw")
    train_file = decompress_unix_compress(root / "raw" / "isolet1+2+3+4.data.Z")
    test_file = decompress_unix_compress(root / "raw" / "isolet5.data.Z")
    train_array = np.loadtxt(train_file, delimiter=",", dtype=np.float32)
    test_array = np.loadtxt(test_file, delimiter=",", dtype=np.float32)
    payload = {
        "x_train": torch.from_numpy(train_array[:, :-1].copy()),
        "y_train": torch.from_numpy(train_array[:, -1].astype(np.int64) - 1),
        "x_test": torch.from_numpy(test_array[:, :-1].copy()),
        "y_test": torch.from_numpy(test_array[:, -1].astype(np.int64) - 1),
        "fidelity_note": "Exact official ISOLET 1-4 training / ISOLET 5 test split.",
    }
    atomic_torch_save(payload, cache)
    return BenchmarkData("isolet", **payload)

def load_sa12():
    root = DATA_DIR / "sa12"
    cache = root / f"processed_seed-{RANDOM_SEED}.pt"
    if cache.exists() and not FORCE_REDOWNLOAD:
        payload = torch.load(cache, map_location="cpu", weights_only=True)
        return BenchmarkData("sa12", **payload)
    archive = download_file(HAPT_URL, root / "hapt.zip")
    safe_extract_zip(archive, root / "raw")
    x_all = torch.from_numpy(np.loadtxt(root / "raw" / "Train" / "X_train.txt", dtype=np.float32))
    y_all = torch.from_numpy(np.loadtxt(root / "raw" / "Train" / "y_train.txt", dtype=np.int64)) - 1
    train_indices, test_indices = stratified_split_exact(y_all, test_size=1_554, seed=RANDOM_SEED)
    payload = {
        "x_train": x_all[train_indices],
        "y_train": y_all[train_indices],
        "x_test": x_all[test_indices],
        "y_test": y_all[test_indices],
        "fidelity_note": (
            "Correct public 7,767-sample, 561-feature, 12-class HAPT/SA12 source and paper counts; "
            "deterministic stratified indices because the paper's random indices are unpublished."
        ),
    }
    atomic_torch_save(payload, cache)
    return BenchmarkData("sa12", **payload)

def load_custom(name):
    path = DATA_DIR / "custom" / f"{name}.npz"
    if not path.exists():
        raise FileNotFoundError(
            f"{name} requires the paper-compatible processed file {path}. "
            "Expected keys: x_train, y_train, x_test, y_test."
        )
    with np.load(path, allow_pickle=False) as arrays:
        required = {"x_train", "y_train", "x_test", "y_test"}
        missing = required - set(arrays.files)
        if missing:
            raise ValueError(f"{path} is missing keys: {sorted(missing)}")
        data = BenchmarkData(
            name,
            torch.as_tensor(arrays["x_train"], dtype=torch.float32),
            torch.as_tensor(arrays["y_train"], dtype=torch.long),
            torch.as_tensor(arrays["x_test"], dtype=torch.float32),
            torch.as_tensor(arrays["y_test"], dtype=torch.long),
            f"User-supplied paper-compatible matrix: {path}",
        )
    return data

LOADERS = {"mnist": load_mnist, "isolet": load_isolet, "sa12": load_sa12}

datasets = {}
for dataset_name in DATASET_NAMES:
    loader = LOADERS.get(dataset_name, lambda name=dataset_name: load_custom(name))
    full_data = loader()
    validate_full_dataset(full_data)
    datasets[dataset_name] = apply_run_limits(full_data)

dataset_summary = pd.DataFrame([
    {
        "dataset": data.name,
        "used_train": len(data.x_train),
        "used_test": len(data.x_test),
        "features": data.n_features,
        "classes": data.n_classes,
        "paper_train": PAPER_DATASETS[data.name]["train"],
        "paper_test": PAPER_DATASETS[data.name]["test"],
        "fidelity": data.fidelity_note,
    }
    for data in datasets.values()
])
display(dataset_summary)


## 5. Project HDC models

These implementations reuse the behavior of the repository's ImageNetV2 HDC heads while accepting arbitrary paper feature vectors:

- feature-wise standardization fitted only on the training split;
- a seeded Gaussian random projection;
- sinusoidal encoding plus similarity-weighted error updates for OnlineHD;
- bipolar projection, real-valued class counts, binary prototypes, and error-driven retraining for QuantHD;
- an optional one-pass bundled-prototype baseline.

Encoding is performed batch-by-batch, so a full `N x 10,000` encoded dataset is never retained in memory.


In [ ]:
STANDARDIZATION_EPSILON = 1e-6

def stable_seed(name, base_seed=RANDOM_SEED):
    digest = hashlib.sha256(name.encode("utf-8")).digest()
    return base_seed + int.from_bytes(digest[:4], "little")

def random_projection(out_features, in_features, seed):
    generator = torch.Generator(device="cpu").manual_seed(seed)
    weight = torch.randn(out_features, in_features, generator=generator)
    return F.normalize(weight, dim=1)

class HDCBase(nn.Module):
    def __init__(self, feature_mean, feature_std, n_classes, dimensions, seed):
        super().__init__()
        self.n_classes = int(n_classes)
        self.dimensions = int(dimensions)
        self.register_buffer("feature_mean", feature_mean.float())
        self.register_buffer("feature_std", feature_std.float().clamp_min(STANDARDIZATION_EPSILON))
        self.register_buffer("projection", random_projection(dimensions, len(feature_mean), seed))

    def standardize(self, features):
        return (features.float() - self.feature_mean) / self.feature_std

    def scores(self, features):
        return self.score_encoded(self.encode(features))

    def predict(self, features):
        return self.scores(features).argmax(dim=1)

class BaselineHDC(HDCBase):
    def __init__(self, feature_mean, feature_std, n_classes, dimensions, seed):
        super().__init__(feature_mean, feature_std, n_classes, dimensions, seed)
        self.register_buffer("prototypes", torch.zeros(n_classes, dimensions))

    def encode(self, features):
        standardized = F.normalize(self.standardize(features), dim=1)
        encoded = F.linear(standardized, self.projection).sign()
        return torch.where(encoded == 0, torch.ones_like(encoded), encoded)

    def score_encoded(self, encoded):
        return F.normalize(encoded, dim=1) @ F.normalize(self.prototypes, dim=1).T

class ProjectOnlineHD(HDCBase):
    def __init__(self, feature_mean, feature_std, n_classes, dimensions, seed):
        super().__init__(feature_mean, feature_std, n_classes, dimensions, seed)
        generator = torch.Generator(device="cpu").manual_seed(seed + 1)
        self.register_buffer("phase", torch.rand(1, dimensions, generator=generator) * (2 * math.pi))
        self.register_buffer("prototypes", torch.zeros(n_classes, dimensions))

    def encode(self, features):
        projected = F.linear(self.standardize(features), self.projection)
        return torch.cos(projected + self.phase) * torch.sin(projected)

    def score_encoded(self, encoded):
        return F.normalize(encoded, dim=1) @ F.normalize(self.prototypes, dim=1).T

class ProjectQuantHD(HDCBase):
    def __init__(self, feature_mean, feature_std, n_classes, dimensions, seed):
        super().__init__(feature_mean, feature_std, n_classes, dimensions, seed)
        self.register_buffer("model_count", torch.zeros(n_classes, dimensions))
        self.register_buffer("quantized_prototypes", torch.ones(n_classes, dimensions, dtype=torch.int8))

    def encode(self, features):
        standardized = F.normalize(self.standardize(features), dim=1)
        encoded = F.linear(standardized, self.projection).sign()
        return torch.where(encoded == 0, torch.ones_like(encoded), encoded)

    def score_encoded(self, encoded):
        prototypes = self.quantized_prototypes.to(dtype=encoded.dtype)
        return (encoded @ prototypes.T) / self.dimensions

    @torch.no_grad()
    def binarize(self):
        quantized = self.model_count.sign()
        quantized = torch.where(quantized == 0, torch.ones_like(quantized), quantized)
        self.quantized_prototypes.copy_(quantized.to(torch.int8))

MODEL_CLASSES = {
    "baseline_hdc": BaselineHDC,
    "onlinehd": ProjectOnlineHD,
    "quanthd": ProjectQuantHD,
}


## 6. Training and evaluation

Full paper-mode runs can be long, especially 40 QuantHD retraining passes over MNIST or a custom PAMAP2 matrix. Compatible checkpoints are reused by default. Test data are never used for fitting standardization, updates, stopping, or checkpoint selection.


In [ ]:
def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elif DEVICE.type == "mps":
        torch.mps.synchronize()

def iter_batches(x, y, batch_size, shuffle, seed):
    generator = torch.Generator(device="cpu").manual_seed(seed)
    loader = DataLoader(
        TensorDataset(x, y),
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda",
    )
    for features, targets in loader:
        yield (
            features.to(DEVICE, non_blocking=True),
            targets.to(DEVICE, non_blocking=True),
        )

@torch.inference_mode()
def accuracy(model, x, y):
    correct = 0
    total = 0
    for features, targets in iter_batches(x, y, BATCH_SIZE, False, RANDOM_SEED):
        predictions = model.predict(features)
        correct += int((predictions == targets).sum().item())
        total += int(targets.numel())
    return correct / total

@torch.no_grad()
def train_baseline(model, data):
    model.prototypes.zero_()
    for features, targets in tqdm(
        iter_batches(data.x_train, data.y_train, BATCH_SIZE, True, RANDOM_SEED),
        total=math.ceil(len(data.y_train) / BATCH_SIZE),
        desc=f"{data.name} baseline bundling",
    ):
        model.prototypes.index_add_(0, targets, model.encode(features))
    return [{"pass": 1, "updated_samples": len(data.y_train)}]

@torch.no_grad()
def train_onlinehd(model, data):
    history = []
    for pass_index in range(1, ONLINEHD_PASSES + 1):
        updated = 0
        batches = iter_batches(
            data.x_train, data.y_train, BATCH_SIZE, True, RANDOM_SEED + pass_index
        )
        for features, targets in tqdm(
            batches,
            total=math.ceil(len(data.y_train) / BATCH_SIZE),
            desc=f"{data.name} OnlineHD pass {pass_index}",
        ):
            encoded = model.encode(features)
            similarities = model.score_encoded(encoded)
            predictions = similarities.argmax(dim=1)
            wrong = predictions != targets
            if wrong.any():
                wrong_encoded = encoded[wrong]
                true_labels = targets[wrong]
                predicted_labels = predictions[wrong]
                wrong_similarities = similarities[wrong]
                true_similarity = wrong_similarities.gather(1, true_labels[:, None])
                predicted_similarity = wrong_similarities.gather(1, predicted_labels[:, None])
                true_update = (1.0 - true_similarity) * wrong_encoded
                predicted_update = (predicted_similarity - 1.0) * wrong_encoded
                model.prototypes.index_add_(
                    0, true_labels, true_update, alpha=ONLINEHD_LEARNING_RATE
                )
                model.prototypes.index_add_(
                    0, predicted_labels, predicted_update, alpha=ONLINEHD_LEARNING_RATE
                )
                updated += int(wrong.sum().item())
        history.append({"pass": pass_index, "updated_samples": updated})
    return history

def quant_validation_split(data):
    validation_size = max(data.n_classes, round(len(data.y_train) * QUANTHD_VALIDATION_FRACTION))
    train_indices, validation_indices = stratified_split_exact(
        data.y_train, test_size=validation_size, seed=RANDOM_SEED + 300
    )
    return BenchmarkData(
        data.name,
        data.x_train[train_indices],
        data.y_train[train_indices],
        data.x_test,
        data.y_test,
        data.fidelity_note,
    ), data.x_train[validation_indices], data.y_train[validation_indices]

@torch.no_grad()
def train_quanthd(model, data):
    training_data = data
    validation_x = validation_y = None
    retrain_passes = QUANTHD_RETRAIN_PASSES
    if QUANTHD_USE_METHOD_STOPPING:
        training_data, validation_x, validation_y = quant_validation_split(data)
        retrain_passes = QUANTHD_MAX_METHOD_PASSES

    model.model_count.zero_()
    initial_batches = iter_batches(
        training_data.x_train, training_data.y_train, BATCH_SIZE, True, RANDOM_SEED
    )
    for features, targets in tqdm(
        initial_batches,
        total=math.ceil(len(training_data.y_train) / BATCH_SIZE),
        desc=f"{data.name} QuantHD initial bundling",
    ):
        model.model_count.index_add_(0, targets, model.encode(features))
    model.binarize()

    history = [{"pass": 0, "updated_samples": len(training_data.y_train), "stage": "initial_bundling"}]
    previous_validation_accuracy = None
    for pass_index in range(1, retrain_passes + 1):
        updated = 0
        batches = iter_batches(
            training_data.x_train,
            training_data.y_train,
            BATCH_SIZE,
            True,
            RANDOM_SEED + pass_index,
        )
        for features, targets in tqdm(
            batches,
            total=math.ceil(len(training_data.y_train) / BATCH_SIZE),
            desc=f"{data.name} QuantHD retrain {pass_index}/{retrain_passes}",
        ):
            encoded = model.encode(features)
            predictions = model.score_encoded(encoded).argmax(dim=1)
            wrong = predictions != targets
            if wrong.any():
                wrong_encoded = encoded[wrong]
                model.model_count.index_add_(
                    0, targets[wrong], wrong_encoded, alpha=QUANTHD_LEARNING_RATE
                )
                model.model_count.index_add_(
                    0, predictions[wrong], wrong_encoded, alpha=-QUANTHD_LEARNING_RATE
                )
                updated += int(wrong.sum().item())
        model.binarize()
        record = {"pass": pass_index, "updated_samples": updated, "stage": "retraining"}
        if validation_x is not None:
            validation_accuracy = accuracy(model, validation_x, validation_y)
            record["validation_accuracy"] = validation_accuracy
            if (
                previous_validation_accuracy is not None
                and abs(validation_accuracy - previous_validation_accuracy) < QUANTHD_CONVERGENCE_EPSILON
            ):
                history.append(record)
                break
            previous_validation_accuracy = validation_accuracy
        history.append(record)
    return history

TRAINERS = {
    "baseline_hdc": train_baseline,
    "onlinehd": train_onlinehd,
    "quanthd": train_quanthd,
}


In [ ]:
def checkpoint_config(model_name, data):
    return {
        "run_mode": RUN_MODE,
        "model": model_name,
        "dataset": data.name,
        "features": data.n_features,
        "classes": data.n_classes,
        "used_train": len(data.y_train),
        "dimensions": HYPERVECTOR_DIMENSIONS,
        "batch_size": BATCH_SIZE,
        "seed": RANDOM_SEED,
        "onlinehd_passes": ONLINEHD_PASSES,
        "onlinehd_lr": ONLINEHD_LEARNING_RATE,
        "quanthd_retrain_passes": QUANTHD_RETRAIN_PASSES,
        "quanthd_lr": QUANTHD_LEARNING_RATE,
        "quanthd_method_stopping": QUANTHD_USE_METHOD_STOPPING,
    }

def checkpoint_path(model_name, data):
    return ARTIFACT_DIR / f"{model_name}_{data.name}_d{HYPERVECTOR_DIMENSIONS}_seed{RANDOM_SEED}.pt"

def make_model(model_name, data):
    feature_mean = data.x_train.float().mean(dim=0)
    feature_std = data.x_train.float().std(dim=0, unbiased=False).clamp_min(STANDARDIZATION_EPSILON)
    return MODEL_CLASSES[model_name](
        feature_mean,
        feature_std,
        data.n_classes,
        HYPERVECTOR_DIMENSIONS,
        stable_seed(data.name),
    ).to(DEVICE)

results = []
histories = {}

for dataset_name, data in datasets.items():
    for model_name in MODEL_NAMES:
        print(f"\n=== {dataset_name} / {model_name} ===")
        config = checkpoint_config(model_name, data)
        path = checkpoint_path(model_name, data)
        model = make_model(model_name, data)
        loaded_checkpoint = False
        training_seconds = 0.0

        if REUSE_CHECKPOINTS and path.exists():
            checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
            if checkpoint.get("config") == config:
                model.load_state_dict(checkpoint["model_state_dict"])
                history = checkpoint.get("history", [])
                training_seconds = float(checkpoint.get("training_seconds", 0.0))
                loaded_checkpoint = True
                print(f"Loaded compatible checkpoint: {path}")
            else:
                print(f"Ignoring incompatible checkpoint: {path}")

        if not loaded_checkpoint:
            synchronize()
            started = time.perf_counter()
            history = TRAINERS[model_name](model, data)
            synchronize()
            training_seconds = time.perf_counter() - started
            if SAVE_ARTIFACTS:
                atomic_torch_save(
                    {
                        "model_state_dict": model.state_dict(),
                        "config": config,
                        "history": history,
                        "training_seconds": training_seconds,
                        "created_at_utc": datetime.now(timezone.utc).isoformat(),
                    },
                    path,
                )
                print(f"Saved checkpoint: {path}")

        model.eval()
        synchronize()
        inference_started = time.perf_counter()
        test_accuracy = accuracy(model, data.x_test, data.y_test)
        synchronize()
        inference_seconds = time.perf_counter() - inference_started
        histories[(dataset_name, model_name)] = history
        results.append(
            {
                "run_mode": RUN_MODE,
                "dataset": dataset_name,
                "model": model_name,
                "train_samples": len(data.y_train),
                "test_samples": len(data.y_test),
                "features": data.n_features,
                "classes": data.n_classes,
                "dimensions": HYPERVECTOR_DIMENSIONS,
                "training_seconds": training_seconds,
                "inference_seconds": inference_seconds,
                "test_samples_per_second": len(data.y_test) / inference_seconds,
                "test_accuracy": test_accuracy,
                "checkpoint_reused": loaded_checkpoint,
                "fidelity_note": data.fidelity_note,
            }
        )
        print(f"test_accuracy={test_accuracy:.4%}; inference={inference_seconds:.3f}s")
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
display(results_df)


## 7. Results, paper reference values, and artifacts

QuantHD's published binary accuracies are shown only as reference lines. A gap is expected because this repository uses its projection-based project encoder, while the QuantHD paper uses base/level hypervectors and its own preprocessing. The comparison is meaningful as a same-dataset/same-scale run of the project model, not as bit-for-bit reproduction.


In [ ]:
QUANTHD_PAPER_BINARY_ACCURACY = {
    "isolet": 0.946,
    "sa12": 0.965,
    "pamap2": 0.913,
    "face": 0.946,
    "cardio": 0.953,
    "extra": 0.726,
}

results_df["paper_quanthd_binary_accuracy"] = results_df["dataset"].map(QUANTHD_PAPER_BINARY_ACCURACY)
results_df["accuracy_gap_vs_quanthd_binary"] = np.where(
    results_df["model"].eq("quanthd"),
    results_df["test_accuracy"] - results_df["paper_quanthd_binary_accuracy"],
    np.nan,
)

if SAVE_ARTIFACTS:
    results_path = ARTIFACT_DIR / "paper_dataset_hdc_results.csv"
    results_df.to_csv(results_path, index=False)
    print(f"Saved results: {results_path}")

display(results_df)

if not results_df.empty:
    plot_data = results_df.pivot(index="dataset", columns="model", values="test_accuracy")
    ax = plot_data.plot(kind="bar", figsize=(10, 5), ylim=(0, 1), rot=0)
    ax.set_ylabel("Test accuracy")
    ax.set_title(f"Project HDC models on paper datasets ({RUN_MODE} mode, D={HYPERVECTOR_DIMENSIONS:,})")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()


## 8. Interpretation checklist

Before reporting a number, verify:

1. `run_mode` is `paper`, not `smoke`.
2. `train_samples`, `test_samples`, `features`, and `classes` match the manifest.
3. SA12 is described as a reconstructed deterministic split because the paper does not publish sample indices.
4. Custom PAMAP2/FACE/PECAN/EXTRA files came from the same feature extraction and split used by the paper; shape validation alone cannot prove provenance.
5. Hardware timing is reported for the current PyTorch device and is not compared directly with the papers' Raspberry Pi or Kintex-7 FPGA measurements.
6. The QuantHD paper has an internal protocol discrepancy: Section III-B says a 5% validation set, `epsilon=0.01`, and at most 30 iterations; Section V-B says the reported models were retrained for 40 iterations at `alpha=0.05`. The default follows Section V-B, while the method-stopping protocol is opt-in.


In [ ]:
# Final paper-vs-project comparison. Run all previous cells first.
# QuantHD values are exact values from Table III (binary model).
# OnlineHD publishes its per-dataset values only as bars in Figure 7; these values
# are approximate digitizations and are deliberately labeled as such.
ONLINEHD_PAPER_APPROX_SINGLE_PASS_ACCURACY = {
    "mnist": 0.963,
    "sa12": 0.896,
    "isolet": 0.890,
    "face": 0.958,
    "pamap2": 0.945,
    "pecan": 0.956,
}

paper_reference_rows = []
for dataset_name, reported_accuracy in ONLINEHD_PAPER_APPROX_SINGLE_PASS_ACCURACY.items():
    paper_reference_rows.append({
        "dataset": dataset_name,
        "model": "onlinehd",
        "paper_accuracy": reported_accuracy,
        "paper_value_type": "approximate digitization (about +/-0.5 percentage points)",
        "paper_protocol": "OnlineHD single-pass, D=10,000; Figure 7",
    })
for dataset_name, reported_accuracy in QUANTHD_PAPER_BINARY_ACCURACY.items():
    paper_reference_rows.append({
        "dataset": dataset_name,
        "model": "quanthd",
        "paper_accuracy": reported_accuracy,
        "paper_value_type": "exact reported value",
        "paper_protocol": "QuantHD binary, D=10,000, alpha=0.05, 40 retraining iterations; Table III/Section V-B",
    })
paper_reference_df = pd.DataFrame(paper_reference_rows)

comparison_df = results_df.merge(paper_reference_df, on=["dataset", "model"], how="inner").copy()
if comparison_df.empty:
    print("No overlapping project and paper results. Include onlinehd and/or quanthd in MODEL_NAMES and rerun.")
else:
    comparison_df["paper_train_samples"] = comparison_df["dataset"].map(
        lambda name: PAPER_DATASETS[name]["train"]
    )
    comparison_df["paper_test_samples"] = comparison_df["dataset"].map(
        lambda name: PAPER_DATASETS[name]["test"]
    )
    comparison_df["same_dataset_sizes"] = (
        comparison_df["train_samples"].eq(comparison_df["paper_train_samples"])
        & comparison_df["test_samples"].eq(comparison_df["paper_test_samples"])
    )
    comparison_df["same_dimensions"] = comparison_df["dimensions"].eq(10_000)
    comparison_df["same_protocol"] = False
    comparison_df.loc[comparison_df["model"].eq("onlinehd"), "same_protocol"] = (
        ONLINEHD_PASSES == 1 and np.isclose(ONLINEHD_LEARNING_RATE, 0.035)
    )
    comparison_df.loc[comparison_df["model"].eq("quanthd"), "same_protocol"] = (
        QUANTHD_RETRAIN_PASSES == 40
        and np.isclose(QUANTHD_LEARNING_RATE, 0.05)
        and not QUANTHD_USE_METHOD_STOPPING
    )
    comparison_df["paper_scale_match"] = (
        comparison_df["run_mode"].eq("paper")
        & comparison_df["same_dataset_sizes"]
        & comparison_df["same_dimensions"]
        & comparison_df["same_protocol"]
    )
    comparison_df["accuracy_gap_percentage_points"] = 100 * (
        comparison_df["test_accuracy"] - comparison_df["paper_accuracy"]
    )
    comparison_df["comparison_status"] = np.where(
        comparison_df["paper_scale_match"],
        "paper-scale match",
        "not paper-comparable (smoke/reduced/changed settings)",
    )
    comparison_df.loc[
        comparison_df["paper_scale_match"] & comparison_df["dataset"].eq("sa12"),
        "comparison_status",
    ] = "paper-scale match; authors' SA12 sample indices are unavailable"

    comparison_view = comparison_df[[
        "dataset", "model", "train_samples", "test_samples", "dimensions",
        "test_accuracy", "paper_accuracy", "accuracy_gap_percentage_points",
        "training_seconds", "inference_seconds", "paper_value_type",
        "comparison_status",
    ]].copy()
    comparison_view["our_accuracy_percent"] = 100 * comparison_view.pop("test_accuracy")
    comparison_view["paper_accuracy_percent"] = 100 * comparison_view.pop("paper_accuracy")
    display(comparison_view.sort_values(["dataset", "model"]).reset_index(drop=True))

    matched = comparison_df[comparison_df["paper_scale_match"]]
    if matched.empty:
        print("No paper-scale comparison is available yet. Run with PAPER_BENCHMARK_MODE=paper and the default settings.")
    else:
        comparison_summary = matched.groupby("model", as_index=False).agg(
            datasets_compared=("dataset", "count"),
            our_mean_accuracy=("test_accuracy", "mean"),
            paper_mean_accuracy=("paper_accuracy", "mean"),
            mean_gap_percentage_points=("accuracy_gap_percentage_points", "mean"),
        )
        comparison_summary["our_mean_accuracy"] *= 100
        comparison_summary["paper_mean_accuracy"] *= 100
        display(comparison_summary)

    plot_rows = comparison_df.sort_values(["dataset", "model"]).reset_index(drop=True)
    labels = [f"{row.dataset}\n{row.model}" for row in plot_rows.itertuples()]
    positions = np.arange(len(plot_rows))
    width = 0.38
    fig, ax = plt.subplots(figsize=(max(9, 1.5 * len(plot_rows)), 5.5))
    ax.bar(positions - width / 2, 100 * plot_rows["test_accuracy"], width, label="Our model")
    ax.bar(positions + width / 2, 100 * plot_rows["paper_accuracy"], width, label="Paper")
    for position, row in zip(positions, plot_rows.itertuples()):
        ax.text(
            position,
            101,
            f"{row.accuracy_gap_percentage_points:+.1f} pp",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    ax.set_xticks(positions, labels)
    ax.set_ylabel("Test accuracy (%)")
    ax.set_ylim(0, 107)
    ax.set_title("Our project models vs. OnlineHD and QuantHD paper results")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

    if SAVE_ARTIFACTS:
        comparison_path = ARTIFACT_DIR / "paper_vs_project_comparison.csv"
        comparison_df.to_csv(comparison_path, index=False)
        print(f"Saved comparison: {comparison_path}")

    print(
        "Only accuracy is compared directly. Our absolute training/inference times are shown for context; "
        "the papers used different CPU/FPGA hardware, so their timing and energy numbers are not merged."
    )
